In [1]:
import scarf

scarf.configure_output(level='WARNING', progress=False)

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    'tenx_5K_pbmc_rnaseq',
    destination='scarf_datasets',
    zarr=True,
)
ds = scarf.DataStore(
    f'{dataset}/data.zarr',
    nthreads=4,
    min_features_per_cell=10,
)
ds.filter_cells(
    attrs=['RNA_nCounts', 'RNA_nFeatures'],
    highs=[15000, 4000],
    lows=[1000, 500],
    reset_previous=True,
)
if 'I__hvgs' not in ds.RNA.feats.columns:
    ds.mark_hvgs(min_cells=20, top_n=500, show_plot=False)

In [2]:
normalized = ds.run_normalization(
    feat_key='hvgs',
    update_state=False,
)
pca = ds.run_pca(normalized, dims=15, update_state=False)
ann = ds.build_ann_index(pca, update_state=False)
neighbors_k11 = ds.query_neighbors(ann, k=11, update_state=False)
graph_k11 = ds.build_connectivity_map(neighbors_k11, update_state=False)

In [3]:
neighbors_k15 = ds.query_neighbors(ann, k=15, update_state=False)
graph_k15 = ds.build_connectivity_map(neighbors_k15, update_state=False)

print('normalization reused:', ds.run_normalization(feat_key='hvgs', update_state=False) == normalized)
print('PCA reused:', ds.run_pca(normalized, dims=15, update_state=False) == pca)
print('ANN index reused:', ds.build_ann_index(pca, update_state=False) == ann)
print('neighbors recomputed:', neighbors_k15 != neighbors_k11)
print('graph recomputed:', graph_k15 != graph_k11)

normalization reused: True


PCA reused: True
ANN index reused: True
neighbors recomputed: True
graph recomputed: True


In [4]:
pca_dims20 = ds.run_pca(normalized, dims=20, update_state=False)
ann_dims20 = ds.build_ann_index(pca_dims20, update_state=False)
neighbors_dims20 = ds.query_neighbors(ann_dims20, k=11, update_state=False)
graph_dims20 = ds.build_connectivity_map(neighbors_dims20, update_state=False)

print('PCA recomputed:', pca_dims20 != pca)
print('ANN index recomputed:', ann_dims20 != ann)
print('neighbors recomputed:', neighbors_dims20 != neighbors_k11)
print('graph recomputed:', graph_dims20 != graph_k11)
print('normalization reused:', ds.run_normalization(feat_key='hvgs', update_state=False) == normalized)

PCA recomputed: True
ANN index recomputed: True
neighbors recomputed: True
graph recomputed: True
normalization reused: True


In [5]:
forced = ds.run_normalization(
    feat_key='hvgs',
    update_state=False,
    invalidate_cache=True,
)
status = ds.inspect_artifact(forced)
print('new artifact:', forced != normalized)
print('complete:', status.complete)
print('operation:', status.operation)

new artifact: True
complete: True
operation: run_normalization


In [6]:
lineage = ds.lineage(
    {
        'k11 graph': graph_k11,
        'k15 graph': graph_k15,
    }
)
print(lineage)

ArtifactLineage(outputs=2, artifacts=10, dependencies=13)


In [7]:
print(lineage.to_mermaid())

flowchart LR
    artifact0["RNA / feature_selection | manual_selection | 49e4f1a46d29"]
    artifact1["datastore / cell_selection | filter_cells | 8d16cb0b79c6"]
    artifact2["RNA / normalized | run_normalization | c9ee7d14c3a3"]
    artifact3["RNA / feature_scaling | calculate_feature_scaling | 8f63f6a872ce"]
    artifact4["RNA / reduction | run_pca | e85f27459efa"]
    artifact5["RNA / ann_index | build_ann_index | f84bc8fa6ac6"]
    artifact6["RNA / neighbors | query_neighbors | 36f68bcf268b"]
    artifact7["RNA / connectivity_map | build_connectivity_map | 6fcd774b617b | outputs: k11 graph"]
    artifact8["RNA / neighbors | query_neighbors | 8e2533f33e33"]
    artifact9["RNA / connectivity_map | build_connectivity_map | ed0e2d5b8a5a | outputs: k15 graph"]
    artifact0 -->|"feature_selection"| artifact2
    artifact1 -->|"cell_selection"| artifact2
    artifact2 -->|"normalized"| artifact3
    artifact1 -->|"pca_cell_selection"| artifact4
    artifact2 -->|"normalized"| artifact4
